## Evaluating Frontend CSV

##### Use the frontend to run the chat queries. After completing the chat, download the CSV file containing the results and save its file path in the evaluation notebook. Then run the evaluation using that CSV file.

In [ ]:
# =============================================================================
# GEO RAG EVALUATION — FRONTEND CSV ONLY
# Scope (per architecture): this notebook only ever sees the frontend CSV
# (query, generated_answer, GSE/GSM results) + the gold CSV (query,
# relevant_gse_ids, ideal_answer). It therefore only computes the two metric
# families that need nothing more than that:
#   - Answer Relevancy            (query + generated_answer)
#   - MRR / Precision@K /
#     Recall@K / nDCG@K           (retrieved GSE ids vs gold GSE ids)
#
# Faithfulness, Contextual Relevancy, Constraint Adherence, Contextual
# Precision, and Contextual Recall all require real retrieved evidence/context
# text (Dataset B) and are DELIBERATELY NOT computed here — they belong in the
# separate "evidence/context" notebook you're building next. Bolting them onto
# this CSV would mean feeding a judge fake context, which produces numbers
# that look like real metrics but aren't measuring what they claim to.
#
# Change RESULTS_CSV / GOLD_CSV each run. Everything else is fixed structure.
# =============================================================================

import os, re, json, math
import pandas as pd, numpy as np
from groq import Groq
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# -----------------------------------------------------------------------------
# 0. CONFIG — edit these per run
# -----------------------------------------------------------------------------
RESULTS_CSV = "results/x.csv"                          # <-- change each run
GOLD_CSV    = "results/y.csv"                                    # <-- change each run
MASTER_CSV  = "deepeval_results/master_eval_scores_frontend.csv"     # <-- fixed, always appended to
JUDGE_MODEL = "openai/gpt-oss-120b"

# Frontend CSV's own column names -> internal names used by the pipeline.
COLUMN_MAP = {
    "user_query":                  "query",
    "llm_generated_final_answer":  "generated_answer",
    "result_type":                 "source",       # values: "study" / "sample" — informational only
    "Retrieved gse_id":            "gse_id",
    "Retrieved gsm_id":            "gsm_id",
    "relevance_score":             "relevance_score",
}

ANSWER_RELEVANCY_THRESHOLD = 0.7
IR_KS = (1, 3, 5, 10)

# -----------------------------------------------------------------------------
# 1. JUDGE — Groq-backed DeepEval model (needed only for Answer Relevancy)
# -----------------------------------------------------------------------------
def _safe_json(t):
    if not t:
        return None
    t = re.sub(r"^```(?:json)?|```$", "", t.strip(), flags=re.M).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    m = re.search(r"\{.*\}", t, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None


class GroqJudge(DeepEvalBaseLLM):
    def __init__(self, model: str = JUDGE_MODEL):
        self.model = model
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))

    def load_model(self):
        return self.client

    def get_model_name(self):
        return f"groq/{self.model}"

    def _chat(self, prompt: str, json_mode: bool = False) -> str:
        kwargs = dict(model=self.model, messages=[{"role": "user", "content": prompt}], temperature=0)
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        return self.client.chat.completions.create(**kwargs).choices[0].message.content or ""

    def generate(self, prompt: str, schema=None):
        if schema is None:
            return self._chat(prompt)
        for _ in range(3):
            try:
                raw = self._chat(prompt + "\nReturn ONLY valid JSON.", json_mode=True)
            except Exception:
                raw = self._chat(prompt + "\nReturn ONLY valid JSON.")
            obj = _safe_json(raw)
            if obj is not None:
                try:
                    return schema.model_validate(obj)
                except Exception:
                    pass
        raise ValueError("Judge failed to produce valid JSON after 3 attempts")

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)


judge = GroqJudge()

# -----------------------------------------------------------------------------
# 2. LOAD + NORMALIZE INPUT CSVs
# -----------------------------------------------------------------------------
def _norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())


def load_frontend_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    missing = [c for c in COLUMN_MAP if c not in df.columns]
    if missing:
        raise KeyError(
            f"RESULTS_CSV is missing expected column(s) {missing}. "
            f"Update COLUMN_MAP at the top of this cell to match your schema."
        )
    df = df.rename(columns=COLUMN_MAP)
    df["relevance_score"] = pd.to_numeric(df["relevance_score"], errors="coerce").fillna(0.0)
    # Rank across ALL rows in a query (study + sample together) — a GSE cited
    # only through a "sample" row is still a real retrieval hit and must not
    # be dropped from the ranking used for MRR/Precision/Recall/nDCG.
    df["chunk_rank"] = (
        df.groupby("query")["relevance_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    return df


def load_gold(path: str) -> dict:
    if not os.path.exists(path):
        print(f"[WARN] GOLD_CSV not found at '{path}' — IR metrics will be NaN for every query.")
        return {}
    gold = pd.read_csv(path, dtype=str).fillna("")
    gmap = {}
    for _, r in gold.iterrows():
        gmap[_norm_text(r["query"])] = {
            "relevant": [x.strip() for x in re.split(r"[;,]", r.get("relevant_gse_ids", "")) if x.strip()],
        }
    return gmap


def ranked_gse_for_query(g: pd.DataFrame) -> list[str]:
    """Unique GSE ids in retrieval order, deduped, regardless of study/sample source."""
    seen, ranked = set(), []
    for _, r in g.sort_values("chunk_rank").iterrows():
        gse = r["gse_id"].strip()
        if gse and gse not in seen:
            seen.add(gse)
            ranked.append(gse)
    return ranked

# -----------------------------------------------------------------------------
# 3. DETERMINISTIC IR METRICS
# -----------------------------------------------------------------------------
def ir_metrics(ranked: list[str], relevant: list[str], ks=IR_KS) -> dict:
    rel_set = set(relevant)
    gains = [1 if g in rel_set else 0 for g in ranked]
    dcg = lambda xs: sum(x / math.log2(i + 2) for i, x in enumerate(xs))

    out = {"MRR": next((1 / (i + 1) for i, g in enumerate(gains) if g), 0.0)}
    for k in ks:
        top_k = gains[:k]
        idcg = dcg(sorted(gains, reverse=True)[:k])
        out[f"Recall@{k}"] = sum(top_k) / max(len(rel_set), 1)
        out[f"Precision@{k}"] = sum(top_k) / k
        out[f"nDCG@{k}"] = (dcg(top_k) / idcg) if idcg else 0.0
    return out


def nan_ir_metrics(ks=IR_KS) -> dict:
    out = {"MRR": np.nan}
    for k in ks:
        out[f"Recall@{k}"] = np.nan
        out[f"Precision@{k}"] = np.nan
        out[f"nDCG@{k}"] = np.nan
    return out

# -----------------------------------------------------------------------------
# 4. SCORE EVERY QUERY
# -----------------------------------------------------------------------------
answer_relevancy_metric = AnswerRelevancyMetric(ANSWER_RELEVANCY_THRESHOLD, model=judge, include_reason=True)


def score_query(query: str, answer: str, ranked: list[str], gold_map: dict) -> dict:
    row = {"query": query}

    # --- Answer Relevancy: needs only query + generated_answer, no context ---
    try:
        case = LLMTestCase(input=query, actual_output=answer)
        answer_relevancy_metric.measure(case)
        row["AnswerRelevancy"] = round(float(answer_relevancy_metric.score), 4)
    except Exception as e:
        row["AnswerRelevancy"] = np.nan
        print(f"[WARN] AnswerRelevancy failed for query '{query[:50]}': {e}")

    # --- Retrieval effectiveness: ranked GSE ids vs gold GSE ids ---
    gold_entry = gold_map.get(_norm_text(query))
    if gold_entry is None:
        row.update({k: v for k, v in nan_ir_metrics().items()})
        row["has_gold_labels"] = False
        row["n_relevant_gold"] = np.nan
    else:
        row.update({k: round(v, 4) for k, v in ir_metrics(ranked, gold_entry["relevant"]).items()})
        row["has_gold_labels"] = True
        row["n_relevant_gold"] = len(gold_entry["relevant"])

    row["n_retrieved"] = len(ranked)
    return row

# -----------------------------------------------------------------------------
# 5. RUN
# -----------------------------------------------------------------------------
results_df = load_frontend_csv(RESULTS_CSV)
gold_map = load_gold(GOLD_CSV)

rows = []
for q, g in results_df.groupby("query"):
    answer = next((a for a in g["generated_answer"] if a.strip()), "")
    ranked = ranked_gse_for_query(g)
    rows.append(score_query(q, answer, ranked, gold_map))

new_df = pd.DataFrame(rows)
new_df["results_csv"] = RESULTS_CSV
new_df["gold_csv"] = GOLD_CSV
new_df["judge_model"] = judge.get_model_name()
new_df["timestamp"] = pd.Timestamp.now().isoformat(timespec="seconds")

os.makedirs(os.path.dirname(MASTER_CSV), exist_ok=True)
if os.path.exists(MASTER_CSV):
    master = pd.read_csv(MASTER_CSV)
    start_idx = master["index"].max() + 1 if len(master) else 0
    new_df.insert(0, "index", range(start_idx, start_idx + len(new_df)))
    combined = pd.concat([master, new_df], ignore_index=True, sort=False)
else:
    new_df.insert(0, "index", range(len(new_df)))
    combined = new_df

combined.to_csv(MASTER_CSV, index=False)
print(f"Appended {len(new_df)} row(s) to {MASTER_CSV} (now {len(combined)} total rows).")
new_df

## Evaluating EVALUATION DATASET by backend 

##### First, run the test_backend notebook. Then, after generating the answers using Cell 21, create a JSON file containing the generated results. Load this JSON file into the evaluation notebook and run the evaluation. Each query will require 5–6 Groq API calls. Therefore, for 50 queries, approximately 250–300 API calls will be made, which may take some time and could potentially be affected by rate limits.

In [ ]:
# ============================================================
# FULL EVALUATION CELL — Judge setup + JSON loading + all metrics + CSV save
# ============================================================

import os, re, json, math
import pandas as pd, numpy as np
from groq import Groq
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, AnswerRelevancyMetric, ContextualRelevancyMetric,
    ContextualPrecisionMetric, ContextualRecallMetric, GEval, BaseMetric,
)
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ------------------------------------------------------------
# CONFIG — change these per run
# ------------------------------------------------------------
EVAL_JSONL = "results/q001.jsonl"
GOLD_CSV   = "results/gold2.csv"
MASTER_CSV = "deepeval_results/master_eval_scores.csv"
JUDGE_MODEL = "openai/gpt-oss-120b"   # different family from llama-3.3-70b generator

# ------------------------------------------------------------
# 1) JUDGE SETUP (Groq-backed deepeval LLM wrapper)
# ------------------------------------------------------------
def _safe_json(t):
    if not t: return None
    t = re.sub(r"^```(?:json)?|```$", "", t.strip(), flags=re.M).strip()
    try: return json.loads(t)
    except: pass
    m = re.search(r"\{.*\}", t, re.S)
    try: return json.loads(m.group(0)) if m else None
    except: return None

class GroqJudge(DeepEvalBaseLLM):
    def __init__(self, model=JUDGE_MODEL):
        self.model = model; self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    def load_model(self): return self.client
    def get_model_name(self): return f"groq/{self.model}"
    def _chat(self, p, jm=False):
        k = dict(model=self.model, messages=[{"role":"user","content":p}], temperature=0)
        if jm: k["response_format"] = {"type":"json_object"}
        return self.client.chat.completions.create(**k).choices[0].message.content or ""
    def generate(self, prompt, schema=None):
        if schema is None: return self._chat(prompt)
        for _ in range(3):
            try: raw = self._chat(prompt + "\nReturn ONLY valid JSON.", jm=True)
            except: raw = self._chat(prompt + "\nReturn ONLY valid JSON.")
            o = _safe_json(raw)
            if o is not None:
                try: return schema.model_validate(o)
                except: pass
        raise ValueError("judge JSON failed")
    async def a_generate(self, prompt, schema=None): return self.generate(prompt, schema)

judge = GroqJudge()

# ------------------------------------------------------------
# 2) LOAD EVAL RECORDS (JSONL — one query per line)
# ------------------------------------------------------------
_records = []
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            _records.append(json.loads(line))

# ------------------------------------------------------------
# 3) LOAD GOLD SET (joined by normalized query text)
# ------------------------------------------------------------
_gold = pd.read_csv(GOLD_CSV, dtype=str).fillna("")
_norm = lambda s: re.sub(r"\s+", " ", str(s).strip().lower())
_gmap = {_norm(r["query"]): {
            "relevant": [x.strip() for x in re.split(r"[;,]", r.get("relevant_gse_ids", "")) if x.strip()],
            "ideal": (r.get("ideal_answer", "") or "").strip(),
         } for _, r in _gold.iterrows()}

# ------------------------------------------------------------
# 4) FILTER MATCH HELPER (does a study's metadata satisfy query filters?)
# ------------------------------------------------------------
def _filter_match(metadata: dict, filters: dict) -> bool:
    if not filters:
        return True
    for field, value in filters.items():
        meta_val = _norm((metadata or {}).get(field, ""))
        if not meta_val or _norm(str(value)) not in meta_val:
            return False
    return True

# ------------------------------------------------------------
# 5) REBUILD TEST CASES FROM JSON RECORDS
# ------------------------------------------------------------
_cases = []
for rec in _records:
    q               = rec.get("query", "")
    study_evidence  = rec.get("study_evidence", [])
    sample_evidence = rec.get("sample_evidence", [])
    filters         = (rec.get("query_info", {}) or {}).get("filters", {})

    # retrieval_context = exactly what generate_answer() showed the LLM
    ctx = []
    for study in study_evidence:
        for field_text in (study.get("fields", {}) or {}).values():
            if field_text:
                ctx.append(field_text.strip())
    for sample in sample_evidence:
        shown = sample.get("shown_to_llm", {}) or {}
        parts = [f"{k}: {v}" for k, v in shown.items() if v]
        if parts:
            ctx.append(" | ".join(parts))
    ctx = ctx[:15]

    ranked_gse = [
        item["gse_id"]
        for item in sorted(rec.get("ranked_gse_ids", []), key=lambda x: x.get("rank", 999))
    ]
    filter_flags = [_filter_match(s.get("metadata", {}), filters) for s in study_evidence[:10]]

    gd = _gmap.get(_norm(q), {"relevant": [], "ideal": ""})

    _cases.append(LLMTestCase(
        input=q,
        actual_output=rec.get("answer", ""),
        retrieval_context=ctx,
        expected_output=gd["ideal"] or None,
        metadata={
            "query_id"      : rec.get("query_id", ""),
            "ranked_gse"    : ranked_gse,
            "filter_matches": filter_flags,
            "relevant_gse"  : gd["relevant"],
        }))

# ------------------------------------------------------------
# 6) DETERMINISTIC IR METRICS (no judge)
# ------------------------------------------------------------
def _ir(ranked, rel, ks=(1, 3, 5, 10)):
    rs = set(rel); gains = [1 if g in rs else 0 for g in ranked]
    dcg = lambda xs: sum(x / math.log2(i + 2) for i, x in enumerate(xs))
    out = {"MRR": next((1 / (i + 1) for i, g in enumerate(gains) if g), 0.0)}
    for k in ks:
        t = gains[:k]; idcg = dcg(sorted(gains, reverse=True)[:k])
        out[f"Recall@{k}"] = sum(t) / max(len(rs), 1)
        out[f"Precision@{k}"] = sum(t) / k
        out[f"nDCG@{k}"] = dcg(t) / idcg if idcg else 0.0
    return out

# ------------------------------------------------------------
# 7) JUDGE-BASED + CUSTOM METRICS
# ------------------------------------------------------------
_built_in = [
    FaithfulnessMetric(0.7, model=judge, include_reason=True),
    AnswerRelevancyMetric(0.7, model=judge, include_reason=True),
    ContextualRelevancyMetric(0.7, model=judge, include_reason=True),
]

_constraint_adherence = GEval(
    name="Constraint Adherence",
    criteria=("Penalize the answer if it recommends any GEO dataset whose organism, "
              "tissue, or assay type does NOT match what the query explicitly asked for. "
              "Reward answers that surface only datasets matching every stated constraint."),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT,
                       LLMTestCaseParams.RETRIEVAL_CONTEXT],
    model=judge, threshold=0.7)

class _GroundingPrecision(BaseMetric):
    def __init__(self, threshold=0.7): self.threshold = threshold
    def measure(self, tc):
        f = (tc.metadata or {}).get("filter_matches", [])
        self.score = (sum(1 for x in f if x) / len(f)) if f else 0.0
        self.success = self.score >= self.threshold
        self.reason = f"{sum(1 for x in f if x)}/{len(f)} top studies matched the grounded filter"
        return self.score
    async def a_measure(self, tc, *a, **k): return self.measure(tc)
    def is_successful(self): return self.success
    @property
    def __name__(self): return "Grounding Precision"

_grounding_precision = _GroundingPrecision(0.7)

# ------------------------------------------------------------
# 8) SCORE EVERY CASE — ASSEMBLE ONE FLAT ROW PER QUERY
# ------------------------------------------------------------
_rows = []
for c in _cases:
    row = {"query_id": (c.metadata or {}).get("query_id", ""), "query": c.input}

    for m in _built_in:
        name = m.__class__.__name__.replace("Metric", "")
        try:
            m.measure(c)
            row[name] = round(float(m.score), 4)
        except Exception as e:
            row[name] = np.nan
            print(f"[WARN] {name} failed for query '{c.input[:50]}': {e}")

    if c.expected_output:
        for cls, name in [(ContextualPrecisionMetric, "ContextualPrecision"),
                           (ContextualRecallMetric, "ContextualRecall")]:
            try:
                mm = cls(0.7, model=judge, include_reason=True)
                mm.measure(c)
                row[name] = round(float(mm.score), 4)
            except Exception as e:
                row[name] = np.nan
                print(f"[WARN] {name} failed for query '{c.input[:50]}': {e}")
    else:
        row["ContextualPrecision"] = np.nan
        row["ContextualRecall"] = np.nan

    try:
        _constraint_adherence.measure(c)
        row["ConstraintAdherence"] = round(float(_constraint_adherence.score), 4)
    except Exception as e:
        row["ConstraintAdherence"] = np.nan
        print(f"[WARN] ConstraintAdherence failed for query '{c.input[:50]}': {e}")

    _grounding_precision.measure(c)
    row["GroundingPrecision"] = round(float(_grounding_precision.score), 4)

    rel    = (c.metadata or {}).get("relevant_gse", [])
    ranked = (c.metadata or {}).get("ranked_gse", [])
    ir_scores = _ir(ranked, rel)
    row.update({k: round(v, 4) for k, v in ir_scores.items()})

    row["n_relevant_gold"] = len(rel)
    row["n_retrieved"]     = len(ranked)
    row["eval_jsonl"]      = EVAL_JSONL
    row["gold_csv"]        = GOLD_CSV
    row["judge_model"]     = judge.get_model_name()
    row["timestamp"]       = pd.Timestamp.now().isoformat(timespec="seconds")
    _rows.append(row)

_new_df = pd.DataFrame(_rows)

# ------------------------------------------------------------
# 9) APPEND TO MASTER CSV (running index, never overwrites)
# ------------------------------------------------------------
os.makedirs(os.path.dirname(MASTER_CSV), exist_ok=True)
if os.path.exists(MASTER_CSV):
    _master = pd.read_csv(MASTER_CSV)
    _start_idx = _master["index"].max() + 1 if len(_master) else 0
    _new_df.insert(0, "index", range(_start_idx, _start_idx + len(_new_df)))
    _combined = pd.concat([_master, _new_df], ignore_index=True, sort=False)
else:
    _new_df.insert(0, "index", range(len(_new_df)))
    _combined = _new_df

_combined.to_csv(MASTER_CSV, index=False)
print(f"Appended {len(_new_df)} row(s) to {MASTER_CSV} (now {len(_combined)} total rows).")
_new_df